# Next-Word Prediction with a Small PyTorch Transformer

This notebook trains a tiny Transformer language model in PyTorch on the opening chapters of Jules Verne's *Around the World in Eighty Days*. It uses a causal mask so each position only sees earlier words, and then generates text starting from "Mr. Phileas Fogg".

**How to run:** Open in Google Colab, choose **Runtime > Change runtime type > GPU**, then **Runtime > Run all**.

_Rewritten from my original Colab notebook: `Mr phileas fogg dataset.ipynb`._

## 1. Imports and seed
PyTorch is pre-installed in Colab; no TensorFlow is needed here.

In [ ]:
import math
import re
import urllib.request

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Download the book

In [ ]:
def download_gutenberg_book(url):
    """Download a plain-text book from Project Gutenberg and remove the licence header/footer."""
    raw = urllib.request.urlopen(url).read().decode("utf-8-sig")
    start = raw.find("*** START OF")
    end = raw.find("*** END OF")
    if start != -1 and end != -1:
        raw = raw[raw.find("\n", start) + 1:end]
    return raw.replace("\r\n", "\n").strip()

In [ ]:
book = download_gutenberg_book("https://www.gutenberg.org/cache/epub/103/pg103.txt")
start = book.find("Mr. Phileas Fogg lived")
text = book[start:start + 50000]
words = re.findall(r"[a-z']+|[.,;!?]", text.lower())
print("Words (incl. punctuation):", len(words))
print(" ".join(words[:40]))

## 3. Vocabulary and dataset
Each sample is a window of `SEQ_LEN` tokens; the target is the same window shifted one step to the right.

In [ ]:
vocab = sorted(set(words))
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}
token_ids = torch.tensor([word_to_id[w] for w in words], dtype=torch.long)
SEQ_LEN = 32


class NextWordDataset(Dataset):
    def __init__(self, ids, seq_len):
        self.ids, self.seq_len = ids, seq_len

    def __len__(self):
        return len(self.ids) - self.seq_len

    def __getitem__(self, i):
        chunk = self.ids[i:i + self.seq_len + 1]
        return chunk[:-1], chunk[1:]


dataset = NextWordDataset(token_ids, SEQ_LEN)
val_size = int(0.1 * len(dataset))
train_set = torch.utils.data.Subset(dataset, range(len(dataset) - val_size))
val_set = torch.utils.data.Subset(dataset, range(len(dataset) - val_size, len(dataset)))
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64)
print("Vocabulary:", len(vocab), "| train:", len(train_set), "| val:", len(val_set))

## 4. The model
Embedding + learned position embedding + `TransformerEncoder` with a causal mask, then a linear layer over the vocabulary.

In [ ]:
class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, max_len=SEQ_LEN):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers)
        self.fc = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device)
        h = self.token_emb(x) * math.sqrt(self.d_model) + self.pos_emb(positions)
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
        h = self.encoder(h, mask=mask, is_causal=True)
        return self.fc(h)  # (batch, seq_len, vocab_size)


model = TinyTransformerLM(len(vocab)).to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

## 5. Train and evaluate

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def run_epoch(loader, train=True):
    model.train(train)
    total = 0.0
    with torch.set_grad_enabled(train):
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            logits = model(src)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total += loss.item()
    return total / len(loader)


for epoch in range(1, 16):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch:2d} | train loss {train_loss:.3f} | val loss {val_loss:.3f}")

## 6. Generate text

In [ ]:
@torch.no_grad()
def generate(prompt, n_words=30, temperature=0.8):
    model.eval()
    ids = [word_to_id[w] for w in re.findall(r"[a-z']+|[.,;!?]", prompt.lower()) if w in word_to_id]
    for _ in range(n_words):
        x = torch.tensor([ids[-SEQ_LEN:]], device=device)
        logits = model(x)[0, -1] / temperature
        next_id = torch.multinomial(torch.softmax(logits, dim=-1), 1).item()
        ids.append(next_id)
    return " ".join(id_to_word[i] for i in ids)


print(generate("Mr. Phileas Fogg"))

In [ ]:
torch.save(model.state_dict(), "fogg_transformer.pt")